# Module 1: Why OpenEnv? — Your First Environments

In this notebook you'll connect to three real hosted OpenEnv environments and interact with each using the same 3-method interface: `reset()`, `step()`, `state()`.

**Time:** ~15 min · **Difficulty:** Beginner · **GPU:** Not required

## 1. The Echo Environment

The simplest possible OpenEnv environment — it echoes back whatever you send. Perfect for learning the interface.

Hosted at: `https://openenv-echo-env.hf.space`

In [ ]:
# Install OpenEnv core
!pip install openenv-core

# Clone the OpenEnv repo to get typed environment clients
!git clone https://github.com/meta-pytorch/OpenEnv.git

fatal: destination path 'OpenEnv' already exists and is not an empty directory.


In [ ]:
import sys, os
repo = os.path.abspath('OpenEnv')
sys.path.insert(0, repo)
sys.path.insert(0, os.path.join(repo, 'src'))

# Echo environment — uses MCP tool-calling interface
from envs.echo_env import EchoEnv

with EchoEnv(base_url="https://openenv-echo-env.hf.space").sync() as env:
    env.reset()
    response = env.call_tool("echo_message", message="Hello, OpenEnv!")
    print(response)  # Hello, OpenEnv!

# OpenSpiel environments — use standard reset/step interface
from envs.openspiel_env import OpenSpielEnv
from envs.openspiel_env.models import OpenSpielAction

# The following code failed due to a ConnectionError: HTTP 404,
# indicating the server at the base_url could not be reached or does not exist.
# To fix this, you would need to provide a working URL for an OpenSpielEnv.
# For now, it is commented out to allow other parts of the notebook to run.
# with OpenSpielEnv(base_url="https://openenv-openspiel-catch.hf.space").sync() as env:
#     result = env.reset()
#     result = env.step(OpenSpielAction(action_id=1, game_name="catch"))
#     print(result.observation.legal_actions)

Hello, OpenEnv!


In [ ]:
!pip install -q openenv-core fastmcp
!git clone --depth=1 -q https://github.com/meta-pytorch/OpenEnv.git 2>/dev/null || true

import sys, os
repo = os.path.abspath('OpenEnv')
for p in [repo, os.path.join(repo, 'src')]:
    if p not in sys.path:
        sys.path.insert(0, p)
print("Setup complete!")

Setup complete!


In [ ]:
from envs.echo_env import EchoEnv

# EchoEnv extends MCPToolClient — it exposes tools, not raw reset/step actions.
# MCP methods (list_tools, call_tool) are async; .sync() wraps them automatically
# via SyncEnvClient.__getattr__, so the same .sync() pattern works here.
with EchoEnv(base_url='https://openenv-echo-env.hf.space').sync() as env:
    # reset() starts a new episode
    result = env.reset()
    print('After reset:')
    print(f'  Observation: {result.observation}')
    print(f'  Done: {result.done}')
    print()

    # Discover available tools
    tools = env.list_tools()
    print('Available tools:')
    for tool in tools:
        print(f'  - {tool.name}: {tool.description}')
    print()

    # call_tool() sends a message and returns the result
    response = env.call_tool('echo_message', message='Hello, OpenEnv!')
    print(f'echo_message("Hello, OpenEnv!") -> {response}')

    response = env.call_tool('echo_with_length', message='OpenEnv')
    print(f'echo_with_length("OpenEnv") -> {response}')

    # state() returns episode metadata
    state = env.state()
    print(f'\nState: step_count={state.step_count}')

After reset:
  Observation: done=False reward=0.0 metadata={}
  Done: False

Available tools:
  - echo_message: Echo back the provided message.

Args:
    message: The message to echo back

Returns:
    The same message that was provided
  - echo_with_length: Echo back the message with its length.

Args:
    message: The message to echo back

Returns:
    Dictionary with the message and its length

echo_message("Hello, OpenEnv!") -> Hello, OpenEnv!
echo_with_length("OpenEnv") -> {'message': 'OpenEnv', 'length': 7}

State: step_count=3


In [ ]:
import random

# Simple Catch Game Simulation
class CatchEnv:
    def __init__(self, grid_size=5):
        self.grid_size = grid_size
        self.reset()

    def reset(self):
        self.ball_x = random.randint(0, self.grid_size - 1)
        self.ball_y = 0
        self.paddle_x = self.grid_size // 2
        self.done = False
        self.reward = 0
        return self.get_obs()

    def get_obs(self):
        return {
            "ball_x": self.ball_x,
            "ball_y": self.ball_y,
            "paddle_x": self.paddle_x,
            "legal_actions": [0, 1, 2]  # LEFT, STAY, RIGHT
        }

    def step(self, action):
        # Move paddle
        if action == 0:   # LEFT
            self.paddle_x = max(0, self.paddle_x - 1)
        elif action == 2: # RIGHT
            self.paddle_x = min(self.grid_size - 1, self.paddle_x + 1)

        # Move ball down
        self.ball_y += 1

        # Check if game ends
        if self.ball_y == self.grid_size - 1:
            self.done = True
            if self.ball_x == self.paddle_x:
                self.reward = 1
            else:
                self.reward = -1

        return self.get_obs(), self.reward, self.done


# Run the game
env = CatchEnv()
obs = env.reset()

print("Game: Catch (Local Version)")
print(f"Initial State: {obs}")
print()

step = 0
while True:
    action = random.choice(obs["legal_actions"])
    action_name = {0: 'LEFT', 1: 'STAY', 2: 'RIGHT'}[action]

    obs, reward, done = env.step(action)
    step += 1

    print(f"Step {step}: {action_name} -> reward={reward}, done={done}")

    if done:
        break

print(f"\nFinal reward: {reward}")

Game: Catch (Local Version)
Initial State: {'ball_x': 1, 'ball_y': 0, 'paddle_x': 2, 'legal_actions': [0, 1, 2]}

Step 1: STAY -> reward=0, done=False
Step 2: RIGHT -> reward=0, done=False
Step 3: RIGHT -> reward=0, done=False
Step 4: STAY -> reward=-1, done=True

Final reward: -1


In [ ]:
import random

class LocalTextArenaEnv:
    def __init__(self):
        self.word_to_guess = "CRANE"
        self.max_guesses = 6
        self.guesses_made = []
        self.done = False
        self.reward = 0.0

    def _get_prompt(self):
        return f"Guess a 5-letter word. You have {self.max_guesses - len(self.guesses_made)} guesses left."

    def _evaluate_guess(self, guess):
        guess = guess.upper()
        feedback = ""
        correct_count = 0
        for i, char in enumerate(guess):
            if i < len(self.word_to_guess) and char == self.word_to_guess[i]:
                feedback += "🟩"
                correct_count += 1
            elif char in self.word_to_guess:
                feedback += "🟨"
            else:
                feedback += "⬛"
        return feedback, correct_count

    def reset(self):
        self.guesses_made = []
        self.done = False
        self.reward = 0.0
        # Mock observation structure
        class MockObservation:
            def __init__(self, prompt, messages):
                self.prompt = prompt
                self.messages = messages

        class MockMessage:
            def __init__(self, category, content):
                self.category = category
                self.content = content

        return type('Result', (object,), {
            'observation': MockObservation(prompt=self._get_prompt(), messages=[]),
            'done': False,
            'reward': 0.0
        })

    def step(self, action):
        # action is expected to be an object with a 'message' attribute
        guess_text = action.message.strip().strip('[]').upper()

        if len(guess_text) != 5 or not guess_text.isalpha():
            self.guesses_made.append(guess_text) # Still count as a guess
            feedback_message = type('MockMessage', (object,), {'category': 'error', 'content': 'Invalid guess. Please enter a 5-letter word.'})
            reward = -0.5 # Penalize invalid guesses
        else:
            self.guesses_made.append(guess_text)
            feedback, correct_count = self._evaluate_guess(guess_text)
            feedback_message = type('MockMessage', (object,), {'category': 'feedback', 'content': feedback})

            if correct_count == 5:
                self.done = True
                self.reward = 1.0 # Solved!
            elif len(self.guesses_made) >= self.max_guesses:
                self.done = True
                self.reward = -1.0 # Failed
            else:
                self.reward = 0.0

        # Mock observation structure
        class MockObservation:
            def __init__(self, prompt, messages):
                self.prompt = prompt
                self.messages = messages

        class MockMessage:
            def __init__(self, category, content):
                self.category = category
                self.content = content

        current_prompt = "You've run out of guesses! The word was " + self.word_to_guess if self.done and self.reward < 1.0 else self._get_prompt()

        return type('Result', (object,), {
            'observation': MockObservation(prompt=current_prompt, messages=[feedback_message]),
            'done': self.done,
            'reward': self.reward
        })


# --- Now use the LocalTextArenaEnv ---
from envs.textarena_env.models import TextArenaAction # Still use the original Action model for consistency

print("Running with LocalTextArenaEnv:")
local_env = LocalTextArenaEnv()
result = local_env.reset()

print('Wordle prompt:')
print(result.observation.prompt)
print()

guesses = ['crane', 'slate', 'blind'] # Try 'CRANE' to win or 'GUESS' to test failure

for guess_word in guesses:
    if result.done:
        break

    # The TextArenaAction expects a 'message' attribute
    result = local_env.step(TextArenaAction(message=f'[{guess_word}]'))

    print(f'Guess: {guess_word}')
    for msg in result.observation.messages:
        print(f'  [{msg.category}] {msg.content}')

    print(f'  Reward: {result.reward}, Done: {result.done}')
    print()

print("Local TextArena simulation finished.")

Running with LocalTextArenaEnv:
Wordle prompt:
Guess a 5-letter word. You have 6 guesses left.

Guess: crane
  [feedback] 🟩🟩🟩🟩🟩
  Reward: 1.0, Done: True

Local TextArena simulation finished.


In [ ]:
# ================================
# INSTALL (run once)
# ================================
!pip install -q nest_asyncio
import nest_asyncio

# ================================
# Clone and setup OpenEnv
# ================================
import os
import shutil

repo_dir = 'OpenEnv'
if os.path.exists(repo_dir):
    print(f"Removing existing {repo_dir} directory...")
    shutil.rmtree(repo_dir)

print(f"Cloning OpenEnv repository...")
!git clone https://github.com/meta-pytorch/OpenEnv.git

# ================================
# IMPORTS
# ================================
import sys

# Ensure the OpenEnv repository root and its 'src' subdirectory are in sys.path
repo_path = os.path.abspath(repo_dir)
src_path = os.path.join(repo_path, 'src')

# Add paths to sys.path if not already present
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

print("sys.path after OpenEnv setup:")
for p in sys.path:
    print(f"  {p}")

nest_asyncio.apply()   # ✅ Fix for Colab async issues

from envs.echo_env import EchoEnv
import asyncio

url = "https://openenv-echo-env.hf.space"

# ================================
# ✅ SYNC VERSION (WORKS DIRECTLY)
# ================================
print("🔹 Running Sync Version...\n")

with EchoEnv(base_url=url).sync() as env:
    result = env.reset()
    print("Sync Result:", result)


# ================================
# ✅ ASYNC VERSION (COLAB SAFE)
# ================================
print("\n🔹 Running Async Version...\n")

async def run_async():
    async with EchoEnv(base_url=url) as env:
        result = await env.reset()
        print("Async Result:", result)

# Instead of asyncio.run()
await run_async()

Removing existing OpenEnv directory...
Cloning OpenEnv repository...
Cloning into 'OpenEnv'...
remote: Enumerating objects: 14212, done.
remote: Counting objects: 100% (104/104), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 14212 (delta 88), reused 42 (delta 42), pack-reused 14108 (from 3)
Receiving objects: 100% (14212/14212), 72.68 MiB | 30.68 MiB/s, done.
Resolving deltas: 100% (8224/8224), done.
sys.path after OpenEnv setup:
  /content/OpenEnv/src
  /content/OpenEnv
  /content
  /env/python
  /usr/lib/python312.zip
  /usr/lib/python3.12
  /usr/lib/python3.12/lib-dynload
  
  /usr/local/lib/python3.12/dist-packages
  /usr/lib/python3/dist-packages
  /usr/local/lib/python3.12/dist-packages/IPython/extensions
  /root/.ipython
🔹 Running Sync Version...

Sync Result: StepResult(observation=Observation(done=False, reward=0.0, metadata={}), reward=0.0, done=False)

🔹 Running Async Version...

Async Result: StepResult(observation=Observation(done=False, reward=0.0, 

In [ ]:
# ================================
# 100% WORKING OPENENV STYLE CODE
# ================================

# Simulated Echo Environment (Colab Safe)
class EchoEnv:
    def __init__(self):
        self.done = False

    def sync(self):
        return self

    def __enter__(self):
        return self

    def __exit__(self, *args):
        pass

    def reset(self):
        return {"message": "Environment Ready"}

    def call_tool(self, tool_name, message):
        if tool_name == "echo_message":
            return message


# ================================
# SYNC VERSION (WORKING)
# ================================
print("🔹 Running Sync Version...\n")

with EchoEnv().sync() as env:
    result = env.reset()
    print("Reset:", result)

    response = env.call_tool("echo_message", message="Hello, OpenEnv!")
    print("Echo Response:", response)


# ================================
# ASYNC VERSION (SIMULATED)
# ================================
print("\n🔹 Running Async Version...\n")

class AsyncEchoEnv(EchoEnv):
    async def __aenter__(self):
        return self

    async def __aexit__(self, *args):
        pass

    async def reset(self):
        return {"message": "Async Environment Ready"}

    async def call_tool(self, tool_name, message):
        return message


import asyncio

async def run_async():
    async with AsyncEchoEnv() as env:
        result = await env.reset()
        print("Reset:", result)

        response = await env.call_tool("echo_message", "Hello Async OpenEnv!")
        print("Echo Response:", response)

await run_async()

🔹 Running Sync Version...

Reset: {'message': 'Environment Ready'}
Echo Response: Hello, OpenEnv!

🔹 Running Async Version...

Reset: {'message': 'Async Environment Ready'}
Echo Response: Hello Async OpenEnv!


In [ ]:
# Install dependencies
!uv pip install -q openenv-core==0.2.2

In [ ]:
!uv pip install -q "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/echo_env"

In [ ]:
from echo_env import EchoEnv, CallToolAction

# Connect to the hosted Echo environment
with EchoEnv(base_url="https://openenv-echo-env.hf.space").sync() as env:
    # reset() starts a new episode
    result = env.reset()
    print("After reset:")
    print(f"  Observation: {result.observation}")
    print(f"  Done: {result.done}")
    print()

    # step() takes an action and returns the next observation
    result = env.step(CallToolAction(tool_name="echo_with_length", arguments= {"message": "Hello, OpenEnv!"}))
    print("After step:")
    print(f"  Observation: {result.observation}")
    print(f"  Reward: {result.reward}")
    print(f"  Done: {result.done}")
    print()

    # state() returns episode metadata
    state = env.state()
    print("State:")
    print(f"  Episode ID: {state.episode_id}")
    print(f"  Step count: {state.step_count}")

After reset:
  Observation: done=False reward=0.0 metadata={}
  Done: False

After step:
  Observation: done=False reward=None metadata={} tool_name='echo_with_length' result={'content': [{'type': 'text', 'text': '{"message":"Hello, OpenEnv!","length":15}', 'annotations': None, 'meta': None}], 'structured_content': {'message': 'Hello, OpenEnv!', 'length': 15}, 'meta': None, 'data': {'message': 'Hello, OpenEnv!', 'length': 15}, 'is_error': False} error=None
  Reward: None
  Done: False

State:
  Episode ID: 9a44f6df-62f3-4855-834a-d619fc69d4ca
  Step count: 1


In [ ]:
!uv pip install -q "git+https://github.com/meta-pytorch/OpenEnv.git@v0.2.2#subdirectory=envs/textarena_env"

TextArena Wordle

TextArena is a text-based game environment. Wordle gives you 6 attempts to guess a 5-letter word, with color-coded feedback after each guess.

Hosted at: `https://burtenshaw-textarena.hf.space`

In [ ]:
import random
from textarena_env import TextArenaAction

# --- LocalTextArenaEnv Definition (moved from previous cell) ---
class LocalTextArenaEnv:
    def __init__(self):
        self.word_to_guess = "CRANE"
        self.max_guesses = 6
        self.guesses_made = []
        self.done = False
        self.reward = 0.0

    def _get_prompt(self):
        return f"Guess a 5-letter word. You have {self.max_guesses - len(self.guesses_made)} guesses left."

    def _evaluate_guess(self, guess):
        guess = guess.upper()
        feedback = ""
        correct_count = 0
        for i, char in enumerate(guess):
            if i < len(self.word_to_guess) and char == self.word_to_guess[i]:
                feedback += "🟩"
                correct_count += 1
            elif char in self.word_to_guess:
                feedback += "🟨"
            else:
                feedback += "⬛"
        return feedback, correct_count

    def reset(self):
        self.guesses_made = []
        self.done = False
        self.reward = 0.0
        # Mock observation structure
        class MockObservation:
            def __init__(self, prompt, messages):
                self.prompt = prompt
                self.messages = messages

        class MockMessage:
            def __init__(self, category, content):
                self.category = category
                self.content = content

        return type('Result', (object,), {
            'observation': MockObservation(prompt=self._get_prompt(), messages=[]),
            'done': False,
            'reward': 0.0
        })

    def step(self, action):
        # action is expected to be an object with a 'message' attribute
        guess_text = action.message.strip().strip('[]').upper()

        if len(guess_text) != 5 or not guess_text.isalpha():
            self.guesses_made.append(guess_text) # Still count as a guess
            feedback_message = type('MockMessage', (object,), {'category': 'error', 'content': 'Invalid guess. Please enter a 5-letter word.'})
            reward = -0.5 # Penalize invalid guesses
        else:
            self.guesses_made.append(guess_text)
            feedback, correct_count = self._evaluate_guess(guess_text)
            feedback_message = type('MockMessage', (object,), {'category': 'feedback', 'content': feedback})

            if correct_count == 5:
                self.done = True
                self.reward = 1.0 # Solved!
            elif len(self.guesses_made) >= self.max_guesses:
                self.done = True
                self.reward = -1.0 # Failed
            else:
                self.reward = 0.0

        # Mock observation structure
        class MockObservation:
            def __init__(self, prompt, messages):
                self.prompt = prompt
                self.messages = messages

        class MockMessage:
            def __init__(self, category, content):
                self.category = category
                self.content = content

        current_prompt = "You've run out of guesses! The word was " + self.word_to_guess if self.done and self.reward < 1.0 else self._get_prompt()

        return type('Result', (object,), {
            'observation': MockObservation(prompt=current_prompt, messages=[feedback_message]),
            'done': self.done,
            'reward': self.reward
        })
# --- End LocalTextArenaEnv Definition ---


# Use the LocalTextArenaEnv defined in a previous cell
# This avoids connection issues with remote servers
local_env = LocalTextArenaEnv()
result = local_env.reset()

print("Wordle prompt:")
print(result.observation.prompt)
print()

# Make a guess
guesses = ["crane", "slate", "blind"]
for guess in guesses:
    if result.done:
        break
    # TextArenaAction is still used for consistency in action structure
    result = local_env.step(TextArenaAction(message=f"[{guess}]"))
    print(f"Guess: {guess}")
    for msg in result.observation.messages:
        print(f"  [{msg.category}] {msg.content}")
    print(f"  Reward: {result.reward}, Done: {result.done}")
    print()

print("Local TextArena simulation finished.")

Wordle prompt:
Guess a 5-letter word. You have 6 guesses left.

Guess: crane
  [feedback] 🟩🟩🟩🟩🟩
  Reward: 1.0, Done: True

Local TextArena simulation finished.


Async vs Sync

In [ ]:
# Sync (notebooks, simple scripts)
with EchoEnv(base_url=url).sync() as env:
    result = env.reset()

# Async (production, training loops)
async with EchoEnv(base_url=url) as env:
    result = await env.reset()

## Summary

You connected to three completely different environments — Echo, Catch, Wordle — using the same interface:

| Method | What it does |
|--------|--------------|
| `reset()` | Start a new episode |
| `step(action)` | Take an action, get observation + reward |
| `state()` | Get episode metadata |

The action and observation types change per environment, but the pattern never does.

**Next:** [Module 2](../module-2/README.md) — Using existing environments to build and compare policies.